# Lab 6 — Linear Regression Evaluation Metrics

**Day 02 · Python for Data Science**

## Goals

1. Train/test split (400/100).
2. Compute R², MSE, MAE, RMSE on held-out data.
3. Compare to mean baseline; preview Ridge/Lasso.

> **Quick check:** train **400** / test **100** · RMSE ≈ **0.69**




**Day 2 flow:** Lab 5 OLS → **Lab 6 (you are here)** metrics → Day 3 classification.

## Metric cheat sheet

| Metric | Better when… |
|--------|-------------|
| R² | → 1 |
| MAE/RMSE | → 0 |

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"
OUTPUT_DIR = GH_ROOT / "hands-on" / "02-python-for-data-science" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

from sklearn.linear_model import LinearRegression

df = pd.read_csv(ZOMATO_CSV)
FEATURES = ["votes", "average_cost_for_two"]
TARGET = "aggregate_rating"

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df[TARGET]
print("Full dataset:", len(df), "rows")


---

## 1. Train/test split

In [ ]:
TEST_SIZE, RANDOM_STATE = 0.2, 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"train: {len(X_train)} | test: {len(X_test)}")
assert len(X_train) == 400 and len(X_test) == 100


---

## 2. Fit and predict

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"intercept: {model.intercept_:.4f}")
print(f"coefficients: {model.coef_.round(6)}")


---

## 3. All four metrics

In [ ]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mse ** 0.5
metrics = pd.DataFrame({"metric": ["R2", "MSE", "MAE", "RMSE"],
                        "value": [r2, mse, mae, rmse]})
display(metrics.round(4))
print(f"RMSE: {rmse:.4f}")


### 3b. Metrics table saved to output

In [ ]:
out = OUTPUT_DIR / "lr_metrics.json"
metrics.round(4).to_json(out, orient="records", indent=2)
print("saved:", out.name)


---

## 4. Visual diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=y_test, y=y_pred, ax=ax, alpha=0.6)
ax.plot([y.min(), y.max()], [y.min(), y.max()], "r--", label="perfect")
ax.set_xlabel("actual"); ax.set_ylabel("predicted")
ax.legend()
plt.tight_layout()
plt.show()

residuals = y_test - y_pred
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(residuals, bins=15, kde=True, ax=ax)
ax.set_title("Test residuals")
plt.tight_layout()
plt.show()


---

## 5. Mean baseline comparison

In [ ]:
baseline = [y_train.mean()] * len(y_test)
print(f"Baseline RMSE: {mean_squared_error(y_test, baseline) ** 0.5:.4f}")
print(f"Model RMSE:    {rmse:.4f}")
print(f"Model R2:      {r2:.4f}")


---

## 6. Ridge and Lasso (L1/L2)




In [ ]:
from sklearn.linear_model import Lasso, Ridge
import numpy as np

rows = []
for name, m in [("OLS", LinearRegression()), ("Ridge", Ridge(1.0)), ("Lasso", Lasso(0.01, max_iter=5000))]:
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    rows.append({"model": name, "rmse": float(np.sqrt(mean_squared_error(y_test, pred)))})
display(pd.DataFrame(rows).round(4))


### 5b. Experiment — test_size 0.3 (optional)

In [ ]:
# Uncomment to explore:
# X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.3, random_state=42)
# m2 = LinearRegression().fit(X_tr2, y_tr2)
# print("30% test RMSE:", (mean_squared_error(y_te2, m2.predict(X_te2)) ** 0.5).round(4))
print("Default: test_size=0.2 → train=400, test=100")


---

## 7. Error by rating tier

In [ ]:
err_df = pd.DataFrame({"actual": y_test, "predicted": y_pred})
err_df["abs_error"] = (err_df["actual"] - err_df["predicted"]).abs()
err_df["tier"] = pd.cut(err_df["actual"], bins=[0, 3, 3.5, 4, 5], labels=["low", "mid", "good", "top"])
display(err_df.groupby("tier", observed=True)["abs_error"].mean().round(3))


---

## 8. Final checkpoint

In [ ]:
assert len(X_train) == 400 and len(X_test) == 100
assert abs(rmse - 0.6852) < 0.02
print("Numbers match — you're good.")
print("\nDay 02 complete — Day 03 introduces Lending Club classification.")



**Previous:** [Lab 5](lab05_linear_regression_fit.ipynb)  
**Next:** Day 03 — Probability & logistic regression